# Study 921 — Bill Ladder vs ETF 🪜

**Does running your own 3-month T-bill ladder beat the cash ETF that charges you to run one?**

A cash ETF holds Treasury bills and rolls them. You can hold Treasury bills and roll them,
at TreasuryDirect, for nothing. So the forum arithmetic says a home-made ladder must beat
BIL by its expense ratio — "free money for ten minutes a quarter".

We **simulate** the ladder: **13 rungs of 91-day bills**, one bought every seven days and
held to maturity, priced off **^IRX** (the 13-week bill discount quote, converted to a
bond-equivalent yield). Say that plainly — the ladder leg is *modelled*, and ^IRX is a
secondary-market quote standing in for the auction stop-out a real buyer would receive, so
the ladder is the arithmetic of a ladder rather than a track record. Only the funds are a
traded tape. We race it against **BIL**, **SGOV** and **SHV** total return over
2007-05-31 → 2026-06-30 (4,799 days, 992 rolls). Cash is the
numeraire, so nothing is excess of anything: the number is the **annualised gap in
basis points a year**. One execution lag — yesterday's quote prices today's purchase.

*Numbers below are the frozen headline (`docs/results.md`, Fingerprint `1e3d76fa1bfa`); the
live cells run the fast offline synthetic control. As-of 2026-06-30.*


## 1. The claim, and why it should be boringly true

Almost every idea on this desk is a story about crowds or risk premia, and almost every one dies. This one is different: it is arithmetic. If two portfolios hold the same Treasury bills and only one of them pays a manager, the other should win by exactly the manager's fee. Nothing to forecast, nothing to be right about.

So the interesting questions are not *does it work* but **how big is it**, **does the tape actually show it**, and **what does it cost you to collect**.

## 2. The tape says yes — by thirteen basis points

Nineteen years of daily data. The ladder wins, and the margin is small enough to fit in a rounding error on a stock chart.

In [1]:
R = dict(bil_gap=12.83, bil_t=2.75, bil_cagr_l=1.4928, bil_cagr_e=1.3616,
         bil_er=13.54, bil_gross=149.7, bil_resid=-0.4, ci_lo=5.32, ci_hi=20.46)
print('home-made ladder : %.4f%% a year' % R['bil_cagr_l'])
print('BIL (the ETF)    : %.4f%% a year' % R['bil_cagr_e'])
print('gap              : %+.2f basis points a year  (HAC t = %+.2f)'
      % (R['bil_gap'], R['bil_t']))
print('95%% bootstrap CI : [%+.2f, %+.2f] bps/yr -- comfortably above zero'
      % (R['ci_lo'], R['ci_hi']))

home-made ladder : 1.4928% a year
BIL (the ETF)    : 1.3616% a year
gap              : +12.83 basis points a year  (HAC t = +2.75)
95% bootstrap CI : [+5.32, +20.46] bps/yr -- comfortably above zero


## 3. And the gap is *exactly* the fee — not a penny more

Here is the cleanest number in the study. BIL charges **13.54 bps** a year. Add that fee back to what BIL actually paid its holders and you get its return *before* fees: **149.7 bps a year**. The ladder earned **149.3 bps a year**.

The difference is **-0.4 basis points a year** — under half a basis point, over nineteen years. There is no cleverness in the ladder, no secret pickup, no skill. It is the fee, and only the fee.

> 🔬 **For the quants** — that residual is the fee identity of Bogle (2014) and Fama-French (2010) measured in the one asset class where security selection has nowhere to hide, so the coefficient on expenses should be exactly −1. It is.

## 4. The tell: the gap does not care what rates are

Split the sample not by date but by the *level* of short rates. When bills yielded an average of **0.15%** the gap was **+12.64 bps**; when they yielded **3.23%** it was **+13.08 bps**. Flat.

That is the signature of a fee. A fund takes its 13.5 basis points whether bills pay five basis points or five per cent. If the ladder were instead earning some clever carry, the gap would grow with the rate level. It doesn't budge.

## 5. Now the bad news — three basis points of friction eats all of it

A 13-rung weekly ladder buys **52 bills a year**. Charge even **1 bp** of round-trip friction on each purchase and the gap falls to **+8.83**; at **3 bps** it is **+0.83** — gone (*t* = +0.18).

Same story if your maturing bill sits in cash before the replacement settles. Three idle days a roll and the gap is **+7.98** (*t* = +1.71).

At TreasuryDirect with auto-reinvestment you pay neither. Through a broker's secondary-market bill desk you pay both, and the whole exercise is a wash.

## 6. And the easy way collects most of it anyway

Race the ladder against **SGOV** — the cheap modern competitor — instead of BIL, and the gap collapses to **+2.76 bps a year** (*t* = +0.62). Buying a cheaper fund captures nearly the whole edge, with none of the 52 auctions.

One more warning. The ladder's measured volatility is **0.15%** against BIL's **0.49%**, which looks like the ladder is safer. It isn't. A bill held to maturity is simply never marked to market — the calm is an accounting convention. Same credit, same maturity, *less* liquid: if you need the money on a Tuesday you sell a bill at whatever the desk quotes, where the ETF holder just hits a bid.

## 7. Live check — the machinery is unbiased (offline synthetic)

We build a fake world with a fake short rate and a fake cash ETF that charges a *known* fee, then run the same ladder pipeline over it. It should recover the fee we planted, and find nothing at all when the fake ETF is free.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from bill_ladder import data, strategy as st
paid, truth = data.synthetic_daily(signal_strength=1.0, seed=921)
free, _     = data.synthetic_daily(signal_strength=0.0, seed=921)
print('planted ETF fee     : %.2f bps/yr' % truth['fee_bps_effective'])
print('ladder recovers     : %+.2f bps/yr  (should match)'
      % st.synthetic_detect(paid)['gap_bps'])
print('free-ETF null       : %+.2f bps/yr  (should be ~0)'
      % st.synthetic_detect(free)['gap_bps'])

planted ETF fee     : 13.50 bps/yr


ladder recovers     : +13.68 bps/yr  (should match)


free-ETF null       : +0.17 bps/yr  (should be ~0)


## Verdict

- **Signal — Real.** The (simulated) ladder really does beat BIL, by **+12.83 bps a year** — significant on a test with nothing to tune (non-overlapping monthly sums, *t* = +3.27), with HAC (+2.75) and the bootstrap [+5.32, +20.46] agreeing — the same sign in all three eras and flat in the level of rates. The gross-of-fee residual of **-0.4 bps** says plainly what it is: the expense ratio, recovered. It is arithmetic that came out right, not an edge anyone discovered.
- **Tradability — Fragile.** Thirteen basis points is the ceiling, and it dies at 3 bps of friction per purchase or 3 idle days per roll. Against SGOV it is already down to +2.76 bps. Buying the cheaper fund gets you almost all of the edge for one click instead of fifty-two — and keeps the liquidity the ladder quietly takes away.